# Dividir un mes de la TLC para cargarlo en BigQuery

La consola de BigQuery solo acepta archivos de **hasta 100 MB** y un mes de viajes pesa unos **500 MB**. Este cuaderno descarga el mes, lo divide en las partes necesarias (cada una de menos de 100 MB) y te las entrega para subirlas a BigQuery.

Ejecuta las celdas en orden con el botón ▶ de cada una.

## 1. Elige tu mes
Cambia el valor por el mes de tu grupo: `2025-01`, `2025-04`, `2025-09` o `2025-12`.

In [ ]:
MES = "2025-06"   # <- cambia aquí el mes de tu grupo

## 2. Descarga el mes desde la página oficial de la TLC
Tarda menos de un minuto: el archivo baja a la máquina de Colab, no a tu PC.

In [ ]:
import os, urllib.request
ARCHIVO = f"fhvhv_tripdata_{MES}.parquet"
url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{ARCHIVO}"
urllib.request.urlretrieve(url, ARCHIVO)
print(f"Descargado {ARCHIVO}: {os.path.getsize(ARCHIVO)/1024/1024:.0f} MB")

## 3. El script que divide
- **Lee** el mes completo con `pyarrow`.
- **Calcula** cuántas partes hacen falta para quedar cerca de 80 MB cada una.
- **Corta por filas** (`slice`) sin cambiar ningún dato y guarda cada parte en Parquet.
- **Comprueba** que ninguna pase de 100 MB (si alguna pasa, corta con una parte más) y que la suma de viajes sea el mes exacto.

In [ ]:
import math
import os
import sys

import pyarrow.parquet as pq

LIMITE_MB = 100          # lo que acepta «Crear tabla -> Subir» en la consola de BigQuery
OBJETIVO_MB = 80         # tamaño buscado por parte: deja margen bajo el límite
MB = 1024 * 1024


def dividir(origen):
    tabla = pq.read_table(origen)                       # el mes completo en memoria
    tam_mb = os.path.getsize(origen) / MB
    base = os.path.basename(origen).replace("fhvhv_tripdata_", "viajes_").replace("-", "_").replace(".parquet", "")
    partes = max(1, math.ceil(tam_mb * 1.10 / OBJETIVO_MB))   # +10 %: al reescribir, cada parte comprime algo peor
    print(f"{origen}: {tabla.num_rows:,} viajes | {tabla.num_columns} columnas | {tam_mb:.0f} MB")

    while True:
        filas = math.ceil(tabla.num_rows / partes)
        archivos = []
        for i in range(partes):
            trozo = tabla.slice(i * filas, filas)                  # filas i*k … i*k+k-1
            destino = f"{base}_parte_{i + 1}.parquet"
            pq.write_table(trozo, destino, compression="snappy")  # misma compresión del original
            archivos.append((destino, trozo.num_rows, os.path.getsize(destino) / MB))
        if max(mb for _, _, mb in archivos) < LIMITE_MB:
            break
        for destino, _, _ in archivos:                             # alguna pasó de 100 MB: una parte más
            os.remove(destino)
        partes += 1

    for destino, n, mb in archivos:
        print(f"  {destino}: {n:,} viajes | {mb:.1f} MB")
    total = sum(n for _, n, _ in archivos)
    print(f"Comprobación: {total:,} viajes en {partes} partes = {tabla.num_rows:,} del mes -> "
          f"{'OK' if total == tabla.num_rows else 'ERROR'}")
    return [d for d, _, _ in archivos]


## 4. Ejecuta la división

In [ ]:
partes = dividir(ARCHIVO)

## 5. Descarga las partes a tu PC
El navegador puede preguntar si permites **descargar varios archivos**: responde *Permitir*. Quedan en tu carpeta de Descargas.

In [ ]:
from google.colab import files
for p in partes:
    files.download(p)